# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vasavijoshi/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method: Random Forest classifier
I chose Random Forest because the refresh opportunity question may depend on nonlinear relationships and interactions between search visibility, engagement, freshness, and content characteristics. The model will predict the probability that a page is declining and use that probability to rank pages for review. I will compare this ranking with my Week-4 rule-based baseline using Precision@K on the same held-out clients. I will also use permutation importance to interpret which features contribute most to the model.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

##Split: grouped by client
I use an 80/20 holdout grouped by client_id, so all pages from a client stay entirely in either the training or test set. This is more honest than a random row split because pages from the same client may share similar characteristics. The test set therefore measures whether the model can rank pages for previously unseen clients.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 Section 2 — grouped client split

import os
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Load the starter dataset
repo_path = "/content/flyrank-ml-internship"

if not os.path.exists(repo_path):
    !git clone https://github.com/Vasavijoshi/flyrank-ml-internship.git

df = pd.read_csv(
    f"{repo_path}/data/raw/content_refresh_anonymized.csv"
)

# Define the outcome used for modeling.
# trend_direction is used ONLY to create the target, never as a feature.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Grouped 80/20 split by client
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        df,
        y=df["is_declining_label"],
        groups=df["client_id"]
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

# Checks
print("Total rows:", len(df))
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTraining clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print("Client overlap:", len(overlap))

print("\nDeclining rate:")
print("Train:", round(train_df["is_declining_label"].mean(), 4))
print("Test :", round(test_df["is_declining_label"].mean(), 4))

Total rows: 30000
Training rows: 23837
Test rows: 6163

Training clients: 25
Test clients: 7
Client overlap: 0

Declining rate:
Train: 0.5501
Test : 0.511


I train the Random Forest only on the training clients and use its predicted probability of decline to rank the test pages. I recreate the Week-4 rule-based score on the same test pages without fitting it to the test outcomes. Both rankings are evaluated with Precision@K, so the comparison measures how many of the highest-priority pages are actually declining.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 Section 3 — Train model and compare with Week-4 baseline

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# Features available before the outcome.
# trend_direction and trend_pct are NOT features.
feature_cols = [
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "content_age_days",
    "word_count",
    "search_volume",
    "cpc"
]

target_col = "is_declining_label"

# Keep only the required columns
X_train = train_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

y_train = train_df[target_col].copy()
y_test = test_df[target_col].copy()

# Median imputation is fitted ONLY on training data
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1,
        min_samples_leaf=5
    ))
])

# Train
model.fit(X_train, y_train)

# Model probability = ranking score
model_prob = model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# Recreate the Week-4 baseline on the SAME test rows
# ---------------------------------------------------------

baseline_test = test_df[
    [
        "content_id",
        "client_id",
        "days_since_last_update",
        "impressions_90d",
        target_col
    ]
].copy()

# Week-4 staleness points
baseline_test["staleness_points"] = np.select(
    [
        baseline_test["days_since_last_update"] <= 90,
        baseline_test["days_since_last_update"] <= 180
    ],
    [
        0,
        1
    ],
    default=2
)

# Week-4 visibility points
baseline_test["visibility_points"] = np.select(
    [
        baseline_test["impressions_90d"] < 250,
        baseline_test["impressions_90d"] < 500
    ],
    [
        0,
        1
    ],
    default=2
)

baseline_test["baseline_score"] = (
    baseline_test["staleness_points"]
    + baseline_test["visibility_points"]
)

# Add model ranking score
baseline_test["model_probability"] = model_prob

# ---------------------------------------------------------
# Precision@K
# ---------------------------------------------------------

def precision_at_k(frame, score_col, k):
    ranked = frame.sort_values(
        [score_col, "impressions_90d"],
        ascending=[False, False]
    ).head(k)

    return ranked[target_col].mean()


k_values = [10, 50, 100]

comparison_rows = []

for k in k_values:
    comparison_rows.append({
        "metric": f"Precision@{k}",
        "Week-4 baseline": precision_at_k(
            baseline_test, "baseline_score", k
        ),
        "Random Forest": precision_at_k(
            baseline_test, "model_probability", k
        )
    })

comparison = pd.DataFrame(comparison_rows)

print("Model feature count:", len(feature_cols))
print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("\nModel vs Week-4 baseline:")
display(comparison.round(4))

Model feature count: 12
Training rows: 23837
Test rows: 6163

Model vs Week-4 baseline:


,metric,Week-4 baseline,Random Forest
0,Precision@10,0.3,0.50
1,Precision@50,0.3,0.64
2,Precision@100,0.3,0.61


The Random Forest performs better than the Week-4 baseline at all tested K values, but some pages are still ranked incorrectly. I examine false positives and false negatives at the default probability threshold and use permutation importance to understand which pre-outcome features the model relies on. This helps identify both model errors and the signals driving the ranking.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-08 Section 4 — Errors and interpretation

from sklearn.inspection import permutation_importance

# ---------------------------------------------------------
# 1. Classification errors at the default 0.50 threshold
# ---------------------------------------------------------

test_analysis = test_df[
    [
        "content_id",
        "client_id",
        "days_since_last_update",
        "impressions_90d",
        "clicks_90d",
        "sessions_90d",
        target_col
    ]
].copy()

test_analysis["predicted_probability"] = model_prob
test_analysis["predicted_label"] = (
    test_analysis["predicted_probability"] >= 0.50
).astype(int)

test_analysis["error_type"] = "Correct"

test_analysis.loc[
    (test_analysis[target_col] == 0) &
    (test_analysis["predicted_label"] == 1),
    "error_type"
] = "False Positive"

test_analysis.loc[
    (test_analysis[target_col] == 1) &
    (test_analysis["predicted_label"] == 0),
    "error_type"
] = "False Negative"

print("Classification error counts:")
print(test_analysis["error_type"].value_counts())

# Show a few false positives
print("\nSample false positives:")
display(
    test_analysis[
        test_analysis["error_type"] == "False Positive"
    ]
    .sort_values("predicted_probability", ascending=False)
    .head(10)
)

# Show a few false negatives
print("\nSample false negatives:")
display(
    test_analysis[
        test_analysis["error_type"] == "False Negative"
    ]
    .sort_values("predicted_probability", ascending=False)
    .head(10)
)

# ---------------------------------------------------------
# 2. Permutation importance
# ---------------------------------------------------------

perm = permutation_importance(
    model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values(
    "importance_mean",
    ascending=False
)

print("\nPermutation importance:")
display(importance.round(4))

Classification error counts:
error_type
Correct           3546
False Positive    1496
False Negative    1121
Name: count, dtype: int64

Sample false positives:


,content_id,client_id,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,is_declining_label,predicted_probability,predicted_label,error_type
26651,content_711c23a1d649,client_f369cb89fc,20,1538,4,6,0,0.936831,1,False Positive
11061,content_0b47dae0c7f9,client_8527a891e2,103,1191,0,5,0,0.926076,1,False Positive
11718,content_1bc2a01f84df,client_f369cb89fc,20,1291,2,3,0,0.925624,1,False Positive
19321,content_c149dfab5d24,client_f369cb89fc,20,2346,3,3,0,0.924860,1,False Positive
22526,content_1d0963b56227,client_4e07408562,104,3445,3,5,0,0.923378,1,False Positive
6718,content_3fbab2102eb5,client_f369cb89fc,20,718,2,7,0,0.913042,1,False Positive
8016,content_c94a53e3bfb8,client_f369cb89fc,20,2164,5,7,0,0.908353,1,False Positive
22812,content_aec0a7f1c9f0,client_f369cb89fc,20,701,0,4,0,0.907235,1,False Positive
27993,content_26d48a980581,client_f369cb89fc,106,1266,0,5,0,0.904725,1,False Positive
4827,content_4d39953789dd,client_f369cb89fc,20,384,1,2,0,0.900999,1,False Positive



Sample false negatives:


,content_id,client_id,days_since_last_update,impressions_90d,clicks_90d,sessions_90d,is_declining_label,predicted_probability,predicted_label,error_type
3809,content_4ab1528abdbd,client_f369cb89fc,20,7403,28,27,1,0.499852,0,False Negative
28942,content_7cb70d02699c,client_e629fa6598,20,200,0,3,1,0.499133,0,False Negative
23528,content_dce9528e18b1,client_4e07408562,7,2519,0,1,1,0.499073,0,False Negative
19455,content_2ed48493988d,client_8527a891e2,20,6,0,1,1,0.499038,0,False Negative
22230,content_7c8916bd0e47,client_4e07408562,7,2864,0,5,1,0.499018,0,False Negative
11662,content_d149b02b1cb1,client_f369cb89fc,20,219,1,2,1,0.499008,0,False Negative
9651,content_3c62618cdd85,client_f369cb89fc,20,28,0,2,1,0.498878,0,False Negative
13026,content_42452f42c008,client_f369cb89fc,20,23,0,2,1,0.498649,0,False Negative
15236,content_e54835b2ecd6,client_f369cb89fc,20,55,0,2,1,0.498357,0,False Negative
20626,content_574f08786f19,client_f369cb89fc,20,33,0,1,1,0.498340,0,False Negative



Permutation importance:


,feature,importance_mean,importance_std
1,impressions_90d,0.0605,0.0031
5,avg_position,0.0350,0.0022
8,content_age_days,0.0114,0.0047
2,clicks_90d,0.0100,0.0017
4,ctr,0.0095,0.0012
7,scroll_rate,0.0044,0.0031
6,engagement_rate,0.0026,0.0008
3,sessions_90d,0.0021,0.0012
11,cpc,-0.0011,0.0007
10,search_volume,-0.0035,0.0023


### Interpretation

The Random Forest improved the ranking over the Week-4 baseline at all tested K values. On the test clients, Precision@10 increased from 0.30 to 0.50, Precision@50 from 0.30 to 0.64, and Precision@100 from 0.30 to 0.61. This suggests that combining several pre-outcome signals produced a more useful ranking than the simple freshness-and-visibility rule.

The error analysis found 1,496 false positives and 1,121 false negatives. The false positives show that some pages received high predicted decline probability even though their observed label was not declining. The false negatives show that some declining pages received probabilities just below the 0.50 classification threshold.

Permutation importance shows that impressions_90d and avg_position were the strongest signals in this model, followed by content_age_days, clicks_90d, and ctr. The negative importance for days_since_last_update means that it did not add useful marginal predictive information after the other features were included in this test; it does not imply that freshness causes better performance. Overall, the model adds value through combining multiple signals rather than simply relying on one rule.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.